# Project 1 - Exploration and model selection

This notebook shows the steps that led to our final model:

1. data loading and exploration,
2. preprocessing (see `preprocessing.py`),
3. comparison of the six basic methods of `implementations.py` (baselines),
4. our model: class-weighted regularized logistic regression trained with Newton's method,
   with hyperparameters chosen by 5-fold cross-validation.

All the models are trained on 60% of the training set and evaluated on the other 40% (hold-out set).
The final predictions are produced by `run.py`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from helpers import load_csv_data, load_feature_names, split_data
from implementations import (
    mean_squared_error_gd,
    mean_squared_error_sgd,
    least_squares,
    ridge_regression,
    logistic_regression,
    reg_logistic_regression,
)
from preprocessing import Preprocessor
from models import (
    balanced_class_weights,
    newton_logistic_regression,
    predict_labels,
    accuracy,
    f1_score,
)
from cross_validation import (
    cross_validation_ridge,
    cross_validation_lambda,
    cross_validation_threshold,
)
from plots import plot_features, plot_cross_validation

## 1. Data loading and exploration

In [ ]:
x, _, y, _, _ = load_csv_data("data")
feature_names = load_feature_names("data")
print(f"{x.shape[0]} samples, {x.shape[1]} features")
print(f"Positive samples (y = 1): {np.mean(y == 1):.2%}")

The classes are very imbalanced (about 9% of positive samples): a model that always predicts -1 has an accuracy of 91% but an F1 score of 0. This is why we use the F1 score to compare models.

In [ ]:
nan_ratio = np.isnan(x).mean(axis=0)
plt.figure(figsize=(5, 3.5))
plt.hist(nan_ratio, bins=20)
plt.xlabel("fraction of missing values (NaN)")
plt.ylabel("number of features")
plt.show()
print(f"Features with >= 95% of NaN: {np.sum(nan_ratio >= 0.95)}")

In [ ]:
# Histogram of every feature (long to display)
# plot_features(x, feature_names)

## 2. Preprocessing

The preprocessing is learned on the training part only and then applied to the hold-out part (see `preprocessing.py` for the details of each step).

In [ ]:
x_tr, x_va, y_tr, y_va = split_data(x, y, ratio=0.6, seed=42)
del x

preprocessor = Preprocessor(
    nan_threshold=0.95, const_threshold=0.99, max_categories=5, corr_threshold=0.99
)
tx_tr = preprocessor.fit_transform(x_tr, feature_names)
tx_va = preprocessor.transform(x_va)
print(f"Features: {x_tr.shape[1]} raw -> {tx_tr.shape[1]} after preprocessing")

## 3. Baselines: the six basic methods

Regression methods predict 1 if the score `tx @ w >= 0`. The logistic methods are trained with labels in {0, 1} and predict 1 if the probability is >= 0.5 (i.e. `tx @ w >= 0`).

In [ ]:
d = tx_tr.shape[1]
y01_tr = (y_tr == 1).astype(float)  # labels in {0, 1} for the logistic methods
# Step sizes: the largest eigenvalue of tx^T tx / N is about 51, so gradient descent
# converges for gamma < 2/51 = 0.039 (MSE) and gamma < 8/51 = 0.156 (logistic loss).
np.random.seed(1)

ridge_lambdas = np.logspace(-4, 0, 7)
best_ridge_lambda, _ = cross_validation_ridge(y_tr, tx_tr, 5, ridge_lambdas)

baselines = {
    "MSE gradient descent": mean_squared_error_gd(y_tr, tx_tr, np.zeros(d), 100, 0.02),
    "MSE SGD": mean_squared_error_sgd(y_tr, tx_tr, np.zeros(d), 10000, 0.001),
    "Least squares": least_squares(y_tr, tx_tr),
    f"Ridge (lambda={best_ridge_lambda:.0e})": ridge_regression(y_tr, tx_tr, best_ridge_lambda),
    "Logistic regression": logistic_regression(y01_tr, tx_tr, np.zeros(d), 100, 0.1),
    "Reg. logistic regression": reg_logistic_regression(y01_tr, tx_tr, 1e-4, np.zeros(d), 100, 0.1),
}
for name, (w, _) in baselines.items():
    y_pred = predict_labels(tx_va, w)
    print(f"{name:28s} F1 = {f1_score(y_va, y_pred):.3f}   accuracy = {accuracy(y_va, y_pred):.3f}")

With the default threshold, all these models predict almost only -1 because of the class imbalance, so their F1 score is very low. We address this with two modifications:

* **balanced class weights**: each sample has weight N / (2 N_class), so both classes count equally in the loss;
* **a tuned decision threshold** on the score `tx @ w`, chosen to maximize the F1 score.

## 4. Our model: weighted regularized logistic regression (Newton's method)

### 4.1 Regularization parameter (5-fold cross-validation, validation logistic loss)

In [ ]:
lambdas = np.logspace(-8, -3, 10)
best_lambda, loss_cv = cross_validation_lambda(y_tr, tx_tr, 5, lambdas, max_iters=10, weighted=True)
print(f"Best lambda: {best_lambda:.2e}")
plot_cross_validation(lambdas, loss_cv, "lambda", "validation logistic loss")

### 4.2 Decision threshold (5-fold cross-validation, validation F1 score)

In [ ]:
thresholds = np.arange(-2, 2, 0.1)
best_threshold, f1_cv = cross_validation_threshold(
    y_tr, tx_tr, 5, best_lambda, thresholds, max_iters=10, weighted=True
)
print(f"Best threshold: {best_threshold:.1f} (cross-validation F1 = {f1_cv.max():.3f})")
plot_cross_validation(thresholds, f1_cv, "threshold on tx @ w", "validation F1 score", log_scale=False)

### 4.3 Evaluation on the hold-out set

In [ ]:
w, _ = newton_logistic_regression(
    y_tr, tx_tr, np.zeros(d), 15, best_lambda, balanced_class_weights(y_tr)
)
y_pred = predict_labels(tx_va, w, best_threshold)
print(f"Hold-out set: F1 = {f1_score(y_va, y_pred):.3f}, accuracy = {accuracy(y_va, y_pred):.3f}")

The final model is retrained on the whole training set with these hyperparameters by `run.py`, which writes the submission file.